# Situação deputado

In [ ]:

try:
    # 1. Apaga o conteúdo atual
    cursor.execute("TRUNCATE TABLE situacao_deputado")

    # 2. Troca a PK de sigla para fk_deputado
    cursor.execute("ALTER TABLE situacao_deputado DROP PRIMARY KEY")
    cursor.execute("ALTER TABLE situacao_deputado ADD PRIMARY KEY (fk_deputado)")

    # 3. Adiciona coluna de peso
    cursor.execute("ALTER TABLE situacao_deputado ADD COLUMN peso DECIMAL(5,2) NULL")

    conn.commit()
    print("✅ Tabela pronta!")

except Exception as e:
    print(f"❌ Erro: {e}")
    conn.rollback()

finally:
    cursor.close()
    conn.close()


✅ Tabela pronta!


In [ ]:

cursor = conn.cursor()

# Corrige a coluna sigla de volta para CHAR(1)
cursor.execute("ALTER TABLE situacao_deputado MODIFY COLUMN sigla CHAR(1) NOT NULL")
conn.commit()

SIGLAS = {
    "Exercício":      "E",
    "Afastado":       "A",
    "Convocado":      "C",
    "Fim de Mandato": "F",
    "Licença":        "L",
    "Suplência":      "S",
    "Suspenso":       "U",
    "Vacância":       "V",
}

def get_deputados():
    deps = []
    pagina = 1
    while True:
        r = requests.get(
            "https://dadosabertos.camara.leg.br/api/v2/deputados",
            params={"pagina": pagina, "itens": 100, "ordem": "ASC", "ordenarPor": "nome"},
            headers={"accept": "application/json"}
        ).json()
        dados = r.get("dados", [])
        if not dados:
            break
        deps.extend(dados)
        pagina += 1
    return deps

def get_situacao(id_dep):
    r = requests.get(
        f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_dep}",
        headers={"accept": "application/json"}
    ).json()
    ultimo = r.get("dados", {}).get("ultimoStatus", {})
    situacao = ultimo.get("situacao", "")
    return {
        "nome":  situacao,
        "sigla": SIGLAS.get(situacao, "?")
    }

deputados = get_deputados()
total = len(deputados)
print(f"Total: {total}")

for i, dep in enumerate(deputados, 1):
    id_dep = dep["id"]
    nome   = dep["nome"]
    print(f"[{i}/{total}] {nome}")

    try:
        sit = get_situacao(id_dep)

        if sit["sigla"]:
            cursor.execute("""
                INSERT INTO situacao_deputado
                    (fk_deputado, nome_deputado, sigla, nome, peso)
                VALUES (%s, %s, %s, %s, NULL)
                ON DUPLICATE KEY UPDATE
                    sigla = VALUES(sigla),
                    nome  = VALUES(nome)
            """, (
                id_dep,
                nome,
                sit["sigla"],
                sit["nome"]
            ))
            conn.commit()

        time.sleep(0.2)

    except Exception as e:
        print(f"  ⚠ Erro no deputado {id_dep}: {e}")
        conn.rollback()

print("✅ Concluído!")
cursor.close()
conn.close()

status proposiçoes

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS proposicao_status (
    id INT AUTO_INCREMENT PRIMARY KEY,
    cd_proposicao INT NOT NULL,
    fk_deputado INT,
    situacao VARCHAR(150),
    categoria VARCHAR(50),
    data_status DATETIME,
    UNIQUE (cd_proposicao)
)
""")

conn.commit()

In [ ]:
import requests
import time

cursor.execute("SELECT cd_proposicao FROM proposicao_status WHERE situacao IS NULL")
ids = cursor.fetchall()
total = len(ids)
print(f"Total para atualizar: {total}")

def classificar(status):
    if status and ("Norma Jurídica" in status or "Aprovado" in status):
        return "Aprovado"
    return "Não aprovado"

for i, item in enumerate(ids, 1):
    id_prop = item[0]

    url = f"https://dadosabertos.camara.leg.br/api/v2/proposicoes/{id_prop}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Erro {id_prop}")
        continue

    dados = response.json()['dados']
    status_info = dados.get('statusProposicao', {})

    situacao = status_info.get('descricaoSituacao')
    data = status_info.get('dataHora')
    categoria = classificar(situacao)

    cursor.execute("""
        UPDATE proposicao_status
        SET situacao = %s,
            categoria = %s,
            data_status = %s
        WHERE cd_proposicao = %s
    """, (situacao, categoria, data, id_prop))

    if i % 100 == 0:
        conn.commit()
        print(f"[{i}/{total}] {id_prop} | {situacao}")

    time.sleep(0.2)

conn.commit()
print("✅ Concluído!")